In [ ]:
(ML_venv_forecast)

In [ ]:
!python --version

Python 3.10.12


## EDA

### Загрузка данных и первичное преобразование

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

In [ ]:
# TODO Пути стоит выносить в отдельные переменные и объявлять их в начале ноутбука
# Так как это константы то будет выглядеть вот так :
SELL_PRICES_PATH = 'C:\Walmart_project_Kuzmenko\data\external\m5-forecasting-accuracy\sell_prices.csv'
SAMPLE_SUBMISSION_PATH = 'C:\Walmart_project_Kuzmenko\data\external\m5-forecasting-accuracy\sample_submission.csv'
CALENDAR_PATH = 'C:\Walmart_project_Kuzmenko\data\external\m5-forecasting-accuracy\calendar.csv'
TRAIN_VALIDATION_PATH = 'C:\Walmart_project_Kuzmenko\data\external\m5-forecasting-accuracy\sales_train_validation.csv'
# sell_prices = pd.read_csv(SELL_PRICES_PATH)
sell_prices = pd.read_csv(SELL_PRICES_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
calendar = pd.read_csv(CALENDAR_PATH)
sales_train_validation = pd.read_csv(TRAIN_VALIDATION_PATH)

In [2]:
SELL_PRICES_PATH = 'sell_prices.csv'
SAMPLE_SUBMISSION_PATH =  'sample_submission.csv'
CALENDAR_PATH = 'calendar.csv'
TRAIN_VALIDATION_PATH = 'sales_train_validation.csv'

sell_prices = pd.read_csv(SELL_PRICES_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
calendar = pd.read_csv(CALENDAR_PATH)
sales_train_validation = pd.read_csv(TRAIN_VALIDATION_PATH)

Выберем магаизн с максимальным оборотом CA_3

In [3]:
sales_train_validation = sales_train_validation[sales_train_validation['store_id'] == 'CA_3']

In [ ]:
# Принтовать датафреймы не очень хорошая практика - так как исчезает красивая таблица которую создает пандас
# Стоит каждый фрейм отображать в отдельной ячейке с помощью df.head() или просто df
sell_prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [ ]:
sample_submission.head()

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [ ]:
sales_train_validation.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
6098,HOBBIES_1_001_CA_3_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_3,CA,0,0,0,0,...,0,2,4,0,1,1,1,0,3,3
6099,HOBBIES_1_002_CA_3_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_3,CA,0,0,0,0,...,0,0,0,2,0,0,0,0,1,0
6100,HOBBIES_1_003_CA_3_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_3,CA,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
6101,HOBBIES_1_004_CA_3_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_3,CA,0,0,0,0,...,2,7,10,15,3,0,3,4,12,3
6102,HOBBIES_1_005_CA_3_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_3,CA,0,0,0,0,...,0,3,5,1,2,0,0,0,5,1


In [ ]:
sales_train_validation.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1',
       'd_2', 'd_3', 'd_4',
       ...
       'd_1904', 'd_1905', 'd_1906', 'd_1907', 'd_1908', 'd_1909', 'd_1910',
       'd_1911', 'd_1912', 'd_1913'],
      dtype='object', length=1919)

In [ ]:
sales_train_validation['store_id'].value_counts()

,count
store_id,
CA_3,3049


Как видно у нас представлены продажи в 4-х магазинах калифорнии, 3-х магазинах Техаса и 3-х магазинах в Висконсине

> лучше это делать с помощью команды `sales_train_validation['store_id'].value_counts()`. Это покажет еще количество уникальных значений, чтобы было понимание относительно распределение данной категориальной фичи + можно визуализировать и построить barplot

In [4]:
data_cols = [col for col in sales_train_validation.columns if 'd_' in col]
not_data_cols = list(set(sales_train_validation.columns) - set(data_cols))

Покажем графики продаж в разных штатах. Для этого сначала преобразуем таблицу с помощью pd.melt() и добавим дату из таблицы calendar

In [5]:
df_sales_train_validation = pd.melt(sales_train_validation, id_vars = not_data_cols, value_vars = data_cols)
df_sales_train_validation = df_sales_train_validation.merge(calendar[['date', 'd']], left_on = ['variable'], right_on = ['d'])
df_sales_train_validation = df_sales_train_validation.drop(columns = ['variable'])
df_sales_train_validation.head(5)

,id,dept_id,store_id,state_id,item_id,cat_id,value,date,d
0,HOBBIES_1_001_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_001,HOBBIES,0,2011-01-29,d_1
1,HOBBIES_1_002_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_002,HOBBIES,0,2011-01-29,d_1
2,HOBBIES_1_003_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_003,HOBBIES,0,2011-01-29,d_1
3,HOBBIES_1_004_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_004,HOBBIES,0,2011-01-29,d_1
4,HOBBIES_1_005_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_005,HOBBIES,0,2011-01-29,d_1


Построение графиков c помощью plotly:

In [ ]:
df_agg = (
    df_sales_train_validation.groupby(["state_id", "store_id", "date"], as_index=False)["value"]
    .sum()
)
unique_states = df_agg['state_id'].unique()
fig = make_subplots(rows=len(unique_states), cols=1, subplot_titles=[f"State: {state}" for state in unique_states])


for i, state in enumerate(unique_states):
    state_data = df_agg[df_agg["state_id"] == state]
    store_ids = state_data["store_id"].unique()

    for store_id in store_ids:
        store_data = state_data[state_data["store_id"] == store_id]

        fig.add_trace(
            go.Scatter(
                x=store_data["date"],
                y=store_data["value"],
                mode="lines+markers",
                name=f"{store_id} (State: {state})"
            ),
            row=i + 1,
            col=1
        )

# Настройка макета
fig.update_layout(
    title="Time Series by Store and State",
    height=400 * len(unique_states),  # Зависит от числа графиков
    showlegend=True
)

# Отображение графика
fig.show()

### Детекция аномалий

#### Медианный фильтр

In [ ]:
df_agg['date'] = pd.to_datetime(df_agg['date'])

In [ ]:
def is_outlier(points, thresh=3.5):
    """
    Returns a boolean array with True if points are outliers and False
    otherwise.

    Parameters:
    -----------
        points : An numobservations by numdimensions array of observations
        thresh : The modified z-score to use as a threshold. Observations with
            a modified z-score (based on the median absolute deviation) greater
            than this value will be classified as outliers.

    Returns:
    --------
        mask : A numobservations-length boolean array.

    References:
    ----------
        Boris Iglewicz and David Hoaglin (1993), "Volume 16: How to Detect and
        Handle Outliers", The ASQC Basic References in Quality Control:
        Statistical Techniques, Edward F. Mykytka, Ph.D., Editor.
    """
    if len(points.shape) == 1:
        points = points[:,None]
    median = np.median(points, axis=0)
    diff = np.sum((points - median)**2, axis=-1)
    diff = np.sqrt(diff)
    med_abs_deviation = np.median(diff)

    modified_z_score = 0.6745 * diff / med_abs_deviation

    return modified_z_score > thresh

In [ ]:
df_agg['anomaly_med'] = is_outlier(np.array(df_agg['value']))

In [ ]:
store_id = "CA_3"
state = "CA"
fig.add_trace(
    go.Scatter(
        x=df_agg["date"],
        y=df_agg["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x = df_agg[df_agg.anomaly_med]['date'],
        y = df_agg[df_agg.anomaly_med]['value'],
        mode="markers",
        marker=dict(color='Black', size=10)
    ),
    row=1,
    col=1,
)
# Настройка макета
fig.update_layout(
    title="Time Series by Store and State",
    height=400 * len(unique_states),  # Зависит от числа графиков
    showlegend=True
)

# Отображение графика
fig.show()

#### Изоляционный лес

In [ ]:
from sklearn.ensemble import IsolationForest

In [ ]:
df_agg = (
    df_sales_train_validation.groupby(["state_id", "store_id", "date"], as_index=False)["value"]
    .sum()
)

In [ ]:
df = df_agg.copy()
df['date'] = pd.to_datetime(df['date'])

# Столбец для хранения аномалий
df['iso_anomaly'] = False

for id in df['store_id'].unique():
    group = df[df['store_id'] == id]
    iso_forest = IsolationForest(contamination=0.01, random_state=42)
    group_values = group[['value']]
    group['iso_anomaly'] = iso_forest.fit_predict(group_values) == -1
    df.loc[group.index, 'iso_anomaly'] = group['iso_anomaly']

In [ ]:
unique_states = df['state_id'].unique()
fig = make_subplots(rows=len(unique_states), cols=1, subplot_titles=[f"State: {state}" for state in unique_states])


for i, state in enumerate(unique_states):
    state_data = df[df["state_id"] == state]
    store_ids = state_data["store_id"].unique()

    for store_id in store_ids:
        store_data = state_data[state_data["store_id"] == store_id]

        fig.add_trace(
            go.Scatter(
                x=store_data["date"],
                y=store_data["value"],
                mode="lines",
                name=f"{store_id} (State: {state})"
            ),
            row=i + 1,
            col=1
        ),
        fig.add_trace(
            go.Scatter(
                x = store_data['date'][store_data['iso_anomaly']],
                y = store_data['value'][store_data['iso_anomaly']],
                mode="markers",
                marker=dict(color='Black', size=10)
            ),
            row=i + 1,
            col=1,
        )

# Настройка макета
fig.update_layout(
    title="Time Series Anomaly by Store and State",
    height=400 * len(unique_states),  # Зависит от числа графиков
    showlegend=True
)

# Отображение графика
fig.show()

### Стационарность рядов

In [ ]:
!pip install statsmodels

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
df_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1913 entries, 0 to 1912
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   state_id     1913 non-null   object        
 1   store_id     1913 non-null   object        
 2   date         1913 non-null   datetime64[ns]
 3   value        1913 non-null   int64         
 4   anomaly_med  1913 non-null   bool          
dtypes: bool(1), datetime64[ns](1), int64(1), object(2)
memory usage: 61.8+ KB


In [ ]:
df_agg = df_agg.drop('anomaly_med', axis = 1)

In [ ]:
df_agg['date'] = pd.to_datetime(df_agg['date'])

adfuler_results_agg = []
ids_agg = []

for id in df_agg['store_id'].unique():
    ids_agg.append(id)
    group = df_agg[df_agg['store_id'] == store_id]
    result = adfuller(group['value'])
    adfuler_results_agg.append("Стационарный" if result[1] < 0.05 else "НЕ стационарный" )

df_adfuler_results_agg = pd.DataFrame({"id": ids_agg, "flag": adfuler_results_agg})

In [ ]:
df_adfuler_results_agg

,id,flag
0,CA_3,НЕ стационарный


## Рaзработка модели

### Base-line: Скользящее среднее

In [ ]:
df_sales_train_validation['date'] = pd.to_datetime(df_sales_train_validation['date'])

In [ ]:
print('минимальная дата:', df_sales_train_validation['date'].min())
print('максимальная дата:', df_sales_train_validation['date'].max())

минимальная дата: 2011-01-29 00:00:00
максимальная дата: 2016-04-24 00:00:00


In [ ]:
train_start_date = '2011-01-29'
train_end_date = '2016-04-18'
val_start_date = '2016-04-18'
val_end_date = '2016-04-24'

In [ ]:
# Разделение на train test
train_df = df_sales_train_validation[
    (df_sales_train_validation['date'] >= train_start_date)
    & (df_sales_train_validation['date'] < train_end_date)
]
val_df = df_sales_train_validation[
    (df_sales_train_validation['date'] >= val_start_date)
    & (df_sales_train_validation['date'] <= val_end_date)
]

In [ ]:
#train_df_test = train_df[train_df['cat_id'] == 'HOBBIES']
#val_df_test = val_df[val_df['cat_id'] == 'HOBBIES']

In [ ]:
from tqdm import tqdm

In [ ]:
import pandas as pd
import numpy as np

class MovingAverageModel:
    def __init__(self, window=7):
        """
        Инициализация модели.
        :param window: Размер окна для вычисления скользящего среднего
        """
        self.window = window
        self.history = {}

    def fit(self, data):
        """
        Обучение модели (сохранение истории продаж).
        :param data: DataFrame с колонками ['item_id', 'date', 'value']
        """
        data['date'] = pd.to_datetime(data['date'])
        for item in tqdm(data['id'].unique()):
            self.history[item] = (
                data[data['id'] == item]
                .sort_values('date')
                ['value']
                .tolist()
            )

    def predict(self, id, horizon=7):
        """
        Прогнозирование продаж.
        :param item_id: ID товара, для которого строится прогноз
        :param horizon: Число дней для предсказания
        :return: Список прогнозных значений
        """
        if id not in self.history:
            raise ValueError(f"Товар {id} не найден в обучающем наборе данных")

        sales = self.history[id]
        if len(sales) < self.window:
            raise ValueError("Недостаточно данных для расчета скользящего среднего")

        predictions = []
        temp_sales = sales.copy()

        for _ in range(horizon):
            moving_avg = np.mean(temp_sales[-self.window:])
            predictions.append(moving_avg)
            temp_sales.append(moving_avg)

        return predictions

In [ ]:
mov_ang_model = MovingAverageModel()
mov_ang_model.fit(train_df)

<ipython-input-60-6808b4b3c5a0>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['date'] = pd.to_datetime(data['date'])
100%|██████████| 3049/3049 [27:53<00:00,  1.82it/s]


In [ ]:
for id in tqdm(train_df['id'].unique()):
    res = mov_ang_model.predict(id, horizon=7)
    val_df.loc[val_df['id'] == id, 'predicted'] = np.array(res)

  0%|          | 0/3049 [00:00<?, ?it/s]<ipython-input-62-713b340a198b>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df.loc[val_df['id'] == id, 'predicted'] = np.array(res)
100%|██████████| 3049/3049 [00:08<00:00, 370.50it/s]


In [ ]:
#Выберем id для демонстрации предсказаний временного ряда
my_index = 'FOODS_2_001_CA_3_validation'
val_df_for_roll = val_df[val_df['date'] >= pd.to_datetime('2016-03-26')]

fig = make_subplots(rows=1, cols=1)

train_df_for_plot = train_df[
            (train_df["date"] >= pd.to_datetime('2016-03-26') - timedelta(days = 100)) &
            (train_df["id"] == my_index)
        ]

val_df_for_plot = val_df_for_roll[val_df_for_roll["id"] == my_index]

store_id = 'CA_3'
state = 'CA'


fig.add_trace(
    go.Scatter(
        x=train_df_for_plot["date"],
        y=train_df_for_plot["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=val_df_for_plot["date"],
        y=val_df_for_plot["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=val_df_for_plot["date"],
        y=val_df_for_plot["predicted"],
        mode="lines+markers",
        name="Predicted"
    ),
    row=1,
    col=1
)

fig.update_layout(
    title=f"Time Series baseline for {my_index}",
    height=400,
    showlegend=True
)

fig.show()

In [46]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, max_error

def smape_score(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def wape_score(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

def evaluate(y_true, y_pred):
    metrics = dict()
    metrics['mae'] = mean_absolute_error(y_true, y_pred)
    metrics['mse'] = mean_squared_error(y_true, y_pred)
    metrics['rmse'] = np.sqrt(metrics['mse'])
    metrics['mape'] = mean_absolute_percentage_error(y_true, y_pred)
    metrics['smape'] = smape_score(y_true, y_pred)
    metrics['max_error'] = max_error(y_true, y_pred)
    metrics['wape'] = wape_score(y_true, y_pred)

    return metrics

def aggregate_metrics(metrics_df):
    summary = {}
    for metric in metrics_df.columns:
        summary[metric] = {
            'mean': metrics_df[metric].mean(),
            '25%': metrics_df[metric].quantile(0.25),
            'median': metrics_df[metric].median(),
            '75%': metrics_df[metric].quantile(0.75)
        }
    return pd.DataFrame(summary)

In [ ]:
metrics_by_id = val_df.groupby('id').apply(lambda group: pd.Series(evaluate(group['value'], group['predicted'])))

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

invalid value encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

invalid value encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

invalid value encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

invalid value encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

invalid value encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

invalid value encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-68-6d13fa4ebe24>:9: R

In [ ]:
val_df_for_roll_unique = val_df.drop_duplicates(subset=['id'])

metrics_by_store = metrics_by_id.groupby(val_df_for_roll_unique.set_index('id')['store_id']).apply(aggregate_metrics)
metrics_by_cat = metrics_by_id.groupby(val_df_for_roll_unique.set_index('id')['cat_id']).apply(aggregate_metrics)


In [ ]:
metrics_by_store

mae       mse      rmse          mape       smape  \
store_id                                                                  
CA_3     mean    1.344594  6.825793  1.644148  1.619997e+15  125.049319   
         25%     0.509975  0.405372  0.636688  1.016723e+00   80.730528   
         median  0.915564  1.249593  1.117852  1.107857e+15  125.465132   
         75%     1.569187  3.820094  1.954506  2.254383e+15  175.197388   

                 max_error      wape  
store_id                              
CA_3     mean     2.968855       inf  
         25%      1.000000  0.696415  
         median   1.959184  1.000000  
         75%      3.509788  1.330432

In [ ]:
metrics_by_cat

mae       mse      rmse          mape       smape  \
cat_id                                                                     
FOODS     mean    1.634851  9.279097  1.982507  1.736927e+15  112.676632   
          25%     0.710997  0.711363  0.843423  1.044967e+00   70.647497   
          median  1.155877  1.925405  1.387590  1.271300e+15  110.127455   
          75%     1.899395  5.178622  2.275659  2.618311e+15  153.561913   
HOBBIES   mean    1.003195  5.388831  1.251168  1.698578e+15  150.077564   
          25%     0.373663  0.214526  0.463170  1.745137e+14  113.214117   
          median  0.605803  0.586074  0.765555  1.030365e+15  162.432440   
          75%     1.050275  1.861697  1.364440  2.128776e+15  193.234254   
HOUSEHOLD mean    1.130451  4.234087  1.391818  1.417107e+15  129.330918   
          25%     0.428571  0.282095  0.531125  7.676206e-01   82.315053   
          median  0.794330  0.944740  0.971977  1.002816e+15  135.648734   
          75%     1.379888  2.764916  1.662804  1.963918e+15  180.457012   

                  max_error      wape  
cat_id                                 
FOODS     mean     3.536350       inf  
          25%      1.428571  0.622529  
          median   2.438062  0.918652  
          75%      4.177851  1.215130  
HOBBIES   mean     2.298271       inf  
          25%      0.769679  0.889481  
          median   1.456952  1.093235  
          75%      2.571429  1.741297  
HOUSEHOLD mean     2.551843       inf  
          25%      0.908843  0.701705  
          median   1.723177  1.000000  
          75%      3.070485  1.406144

### Попробуем SARIMAX

Эта модель должно учитывать и отсутсвие стационарности и сезонность. Так что мы не будем предобрабатывать ряд

In [ ]:
import warnings
warnings.simplefilter(action = 'ignore', category = Warning)

from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
from tqdm import tqdm

In [ ]:
for id in tqdm(train_df['id'].unique()):
    group = train_df[train_df['id'] == id]
    model = SARIMAX(group['value'],
                order = (3, 0, 0),
                seasonal_order = (0, 1, 1, 7)).fit()
    res = model.get_forecast(steps = 7)
    val_df.loc[val_df['id'] == id, 'predicted'] = np.array(res.predicted_mean)


Выходные данные были обрезаны до нескольких последних строк (5000).
An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.

/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

 86%|████████▋ | 2633/3049 [2:15:59<21:11,  3.06s/it]/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.

/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.

/usr/local/lib/

In [ ]:
fig = make_subplots(rows=1, cols=1)

my_index = "FOODS_1_002_CA_3_validation"
train_df_for_plot = train_df[
            (train_df["date"] >= pd.to_datetime('2016-03-26') - timedelta(days = 100)) &
            (train_df["id"] == my_index)
        ]

val_df_for_plot = val_df[val_df["id"] == my_index]
store_id = 'CA_3'
state = 'CA'
fig.add_trace(
    go.Scatter(
        x=train_df_for_plot["date"],
        y=train_df_for_plot["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=val_df_for_plot["date"],
        y=val_df_for_plot["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=val_df_for_plot["date"],
        y=val_df_for_plot["predicted"],
        mode="lines+markers",
        name="Predicted"
    ),
    row=1,
    col=1
)

fig.update_layout(
    title=f"Time Series baseline for {my_index}",
    height=400,
    showlegend=True
)

fig.show()

In [ ]:
metrics_by_id = val_df.groupby('id').apply(lambda group: pd.Series(evaluate(group['value'], group['predicted'])))

In [ ]:
val_df_for_roll_unique = val_df.drop_duplicates(subset=['id'])

metrics_by_store = metrics_by_id.groupby(val_df_for_roll_unique.set_index('id')['store_id']).apply(aggregate_metrics)
metrics_by_cat = metrics_by_id.groupby(val_df_for_roll_unique.set_index('id')['cat_id']).apply(aggregate_metrics)


In [ ]:
metrics_by_store

mae       mse      rmse          mape       smape  \
store_id                                                                  
CA_3     mean    1.319810  6.240063  1.607732  1.746702e+15  127.381848   
         25%     0.518786  0.384203  0.619842  5.962096e+14   82.352196   
         median  0.897659  1.205669  1.098029  1.221794e+15  129.794139   
         75%     1.521131  3.468414  1.862368  2.155758e+15  178.061359   

                 max_error      wape  
store_id                              
CA_3     mean     2.943797       inf  
         25%      0.972907  0.671536  
         median   1.941291  1.007471  
         75%      3.414928  1.582382

In [ ]:
metrics_by_cat

mae       mse      rmse          mape       smape  \
cat_id                                                                     
FOODS     mean    1.603271  8.408243  1.939278  1.979096e+15  113.338324   
          25%     0.721457  0.737408  0.858725  6.120329e+14   70.142774   
          median  1.100263  1.787274  1.336890  1.514759e+15  111.511588   
          75%     1.841176  4.991656  2.234201  2.407784e+15  154.884285   
HOBBIES   mean    0.993781  5.237134  1.238183  1.649023e+15  153.299505   
          25%     0.362055  0.187847  0.433413  6.218537e+14  119.136052   
          median  0.575370  0.546361  0.739162  1.053312e+15  167.858849   
          75%     1.024121  1.633824  1.278211  1.835496e+15  191.230087   
HOUSEHOLD mean    1.106701  3.805471  1.352110  1.480453e+15  132.670356   
          25%     0.445045  0.246789  0.496779  5.444645e+14   85.734512   
          median  0.769509  0.933814  0.966341  1.112716e+15  142.688013   
          75%     1.342340  2.688208  1.639576  1.814820e+15  183.985411   

                  max_error      wape  
cat_id                                 
FOODS     mean     3.502098       inf  
          25%      1.476819  0.603790  
          median   2.394887  0.884195  
          75%      4.096250  1.358532  
HOBBIES   mean     2.334342       inf  
          25%      0.724871  0.922324  
          median   1.420339  1.219621  
          75%      2.475769  2.242090  
HOUSEHOLD mean     2.506416       inf  
          25%      0.861274  0.690685  
          median   1.745673  1.049170  
          75%      3.084528  1.640574

### Prophet

In [ ]:
!pip install prophet

In [ ]:
val_df.to_csv('val_df_arima_pred', sep=',', index=False, encoding='utf-8')


In [ ]:
from prophet import Prophet

In [ ]:
from tqdm import tqdm

In [ ]:
for id in tqdm(train_df['id'].unique()):
    group = train_df[train_df['id'] == id][['date', 'value']]
    group.columns = ['ds', 'y']
    model = Prophet(daily_seasonality=True, mcmc_samples=0)
    model.fit(group)
    future = model.make_future_dataframe(periods=7)
    forecast = model.predict(future)
    val_df.loc[val_df['id'] == id, 'predicted'] = np.array(forecast[forecast['ds'] >= val_start_date]['yhat'])

Выходные данные были обрезаны до нескольких последних строк (5000).
INFO:cmdstanpy:Chain [1] start processing
15:39:24 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
 82%|████████▏ | 2494/3049 [57:31<12:24,  1.34s/it]DEBUG:cmdstanpy:input tempfile: /tmp/tmpqdh6hcix/n42540as.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpqdh6hcix/b8193aju.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.11/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=49619', 'data', 'file=/tmp/tmpqdh6hcix/n42540as.json', 'init=/tmp/tmpqdh6hcix/b8193aju.json', 'output', 'file=/tmp/tmpqdh6hcix/prophet_modeldp5bhe46/prophet_model-20250227153925.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
15:39:25 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
15:39:25 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] 

In [ ]:
fig = make_subplots(rows=1, cols=1)

my_index = "FOODS_1_002_CA_3_validation"
train_df_for_plot = train_df[
            (train_df["date"] >= pd.to_datetime('2016-03-26') - timedelta(days = 100)) &
            (train_df["id"] == my_index)
        ]

val_df_for_plot = val_df[val_df["id"] == my_index]
store_id = 'CA_3'
state = 'CA'
fig.add_trace(
    go.Scatter(
        x=train_df_for_plot["date"],
        y=train_df_for_plot["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=val_df_for_plot["date"],
        y=val_df_for_plot["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=val_df_for_plot["date"],
        y=val_df_for_plot["predicted"],
        mode="lines+markers",
        name="Predicted"
    ),
    row=1,
    col=1
)

fig.update_layout(
    title=f"Time Series baseline for {my_index}",
    height=400,
    showlegend=True
)

fig.show()

In [ ]:
metrics_by_id = val_df.groupby('id').apply(lambda group: pd.Series(evaluate(group['value'], group['predicted'])))

In [ ]:
val_df_for_roll_unique = val_df.drop_duplicates(subset=['id'])

metrics_by_store = metrics_by_id.groupby(val_df_for_roll_unique.set_index('id')['store_id']).apply(aggregate_metrics)
metrics_by_cat = metrics_by_id.groupby(val_df_for_roll_unique.set_index('id')['cat_id']).apply(aggregate_metrics)


In [ ]:
metrics_by_store

mae       mse      rmse          mape       smape  \
store_id                                                                  
CA_3     mean    1.404427  6.926294  1.696909  1.866733e+15  129.663699   
         25%     0.540048  0.410610  0.640789  4.835546e+14   84.149968   
         median  0.916093  1.251308  1.118619  1.225321e+15  133.450237   
         75%     1.584189  3.682935  1.919098  2.206462e+15  181.248061   

                 max_error      wape  
store_id                              
CA_3     mean     3.030480       inf  
         25%      0.999199  0.692372  
         median   1.961743  1.027361  
         75%      3.497105  1.594174

In [ ]:
metrics_by_cat

mae       mse      rmse          mape       smape  \
cat_id                                                                     
FOODS     mean    1.719695  9.289050  2.068028  2.135996e+15  116.472507   
          25%     0.736266  0.780348  0.883373  4.542379e+14   74.304058   
          median  1.164971  1.982946  1.408171  1.450309e+15  113.385945   
          75%     1.924067  5.452097  2.334973  2.539468e+15  159.944280   
HOBBIES   mean    1.006677  5.155876  1.238961  1.710067e+15  154.049494   
          25%     0.391350  0.203775  0.451414  5.471616e+14  122.770721   
          median  0.601525  0.539320  0.734384  1.088808e+15  168.906098   
          75%     1.040624  1.665715  1.290626  1.907262e+15  192.941099   
HOUSEHOLD mean    1.186364  4.638812  1.434677  1.581714e+15  134.609037   
          25%     0.455819  0.262340  0.512190  4.495717e+14   86.412275   
          median  0.780292  0.981216  0.990563  1.103694e+15  142.819069   
          75%     1.379671  2.892757  1.700811  1.899445e+15  186.507635   

                  max_error      wape  
cat_id                                 
FOODS     mean     3.659207       inf  
          25%      1.470842  0.630616  
          median   2.453429  0.920896  
          75%      4.209929  1.369234  
HOBBIES   mean     2.266723       inf  
          25%      0.698416  0.923758  
          median   1.430017  1.197905  
          75%      2.447612  2.263975  
HOUSEHOLD mean     2.579708       inf  
          25%      0.867487  0.693871  
          median   1.744530  1.064820  
          75%      3.087716  1.626758

In [ ]:
val_df.to_csv('val_df_prophet_pred', sep=',', index=False, encoding='utf-8')

### Градиентный бустинг CatBoost

In [6]:
df_sales_train_validation['date'] = pd.to_datetime(df_sales_train_validation['date'])

In [7]:
df_sales_train_validation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5832737 entries, 0 to 5832736
Data columns (total 9 columns):
 #   Column    Dtype         
---  ------    -----         
 0   id        object        
 1   dept_id   object        
 2   store_id  object        
 3   state_id  object        
 4   item_id   object        
 5   cat_id    object        
 6   value     int64         
 7   date      datetime64[ns]
 8   d         object        
dtypes: datetime64[ns](1), int64(1), object(7)
memory usage: 400.5+ MB


In [8]:
df_sales_train_validation['sales_last_week'] = df_sales_train_validation.groupby('id')['value'].shift(7)
df_sales_train_validation['sales_last_month'] = df_sales_train_validation.groupby('id')['value'].shift(30)

In [9]:
df_sales_train_validation['moving_avg_7'] = df_sales_train_validation.groupby('id')['value'].transform(lambda x: x.rolling(window = 7, min_periods = 1).mean())
df_sales_train_validation['moving_avg_30'] = df_sales_train_validation.groupby('id')['value'].transform(lambda x: x.rolling(window = 30, min_periods = 1).mean())

In [10]:
df_sales_train_validation.dropna(inplace=True)

In [13]:
df_sales_train_validation.head(40)

,id,dept_id,store_id,state_id,item_id,cat_id,value,date,d,sales_last_week,sales_last_month,moving_avg_7,moving_avg_30
91470,HOBBIES_1_001_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_001,HOBBIES,0,2011-02-28,d_31,0.0,0.0,0.000000,0.000000
91471,HOBBIES_1_002_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_002,HOBBIES,0,2011-02-28,d_31,0.0,0.0,0.000000,0.000000
91472,HOBBIES_1_003_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_003,HOBBIES,0,2011-02-28,d_31,0.0,0.0,0.000000,0.000000
91473,HOBBIES_1_004_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_004,HOBBIES,0,2011-02-28,d_31,0.0,0.0,0.000000,0.400000
91474,HOBBIES_1_005_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_005,HOBBIES,0,2011-02-28,d_31,0.0,0.0,0.000000,0.000000
91475,HOBBIES_1_006_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_006,HOBBIES,0,2011-02-28,d_31,0.0,0.0,0.000000,0.000000
91476,HOBBIES_1_007_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_007,HOBBIES,0,2011-02-28,d_31,0.0,0.0,0.000000,0.000000
91477,HOBBIES_1_008_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_008,HOBBIES,0,2011-02-28,d_31,47.0,11.0,12.000000,17.366667
91478,HOBBIES_1_009_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_009,HOBBIES,0,2011-02-28,d_31,0.0,3.0,1.142857,1.566667
91479,HOBBIES_1_010_CA_3_validation,HOBBIES_1,CA_3,CA,HOBBIES_1_010,HOBBIES,0,2011-02-28,d_31,0.0,1.0,0.571429,0.633333


In [11]:
df_sales_train_validation.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5741267 entries, 91470 to 5832736
Data columns (total 13 columns):
 #   Column            Dtype         
---  ------            -----         
 0   id                object        
 1   dept_id           object        
 2   store_id          object        
 3   state_id          object        
 4   item_id           object        
 5   cat_id            object        
 6   value             int64         
 7   date              datetime64[ns]
 8   d                 object        
 9   sales_last_week   float64       
 10  sales_last_month  float64       
 11  moving_avg_7      float64       
 12  moving_avg_30     float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(7)
memory usage: 613.2+ MB


In [14]:
df_sales_train_validation.drop(columns= ['id', 'state_id'], inplace=True)

In [16]:
df_sales_train_validation.drop(columns= ['d'], inplace=True)

In [18]:
df_sales_train_validation.drop(columns= ['store_id'], inplace=True)

In [19]:
df_sales_train_validation.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5741267 entries, 91470 to 5832736
Data columns (total 9 columns):
 #   Column            Dtype         
---  ------            -----         
 0   dept_id           object        
 1   item_id           object        
 2   cat_id            object        
 3   value             int64         
 4   date              datetime64[ns]
 5   sales_last_week   float64       
 6   sales_last_month  float64       
 7   moving_avg_7      float64       
 8   moving_avg_30     float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(3)
memory usage: 438.0+ MB


In [20]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 5.5 MB/s eta 0:00:00


In [21]:
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [22]:
# Create additional time-based features
df_sales_train_validation['day_of_week'] = df_sales_train_validation['date'].dt.dayofweek
df_sales_train_validation['month'] = df_sales_train_validation['date'].dt.month
df_sales_train_validation['year'] = df_sales_train_validation['date'].dt.year
df_sales_train_validation.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5741267 entries, 91470 to 5832736
Data columns (total 12 columns):
 #   Column            Dtype         
---  ------            -----         
 0   dept_id           object        
 1   item_id           object        
 2   cat_id            object        
 3   value             int64         
 4   date              datetime64[ns]
 5   sales_last_week   float64       
 6   sales_last_month  float64       
 7   moving_avg_7      float64       
 8   moving_avg_30     float64       
 9   day_of_week       int32         
 10  month             int32         
 11  year              int32         
dtypes: datetime64[ns](1), float64(4), int32(3), int64(1), object(3)
memory usage: 503.7+ MB


In [15]:
train_start_date = '2011-01-29'
train_end_date = '2016-04-18'
val_start_date = '2016-04-18'
val_end_date = '2016-04-24'

In [23]:
# Разделение на train test
train_df = df_sales_train_validation[
    (df_sales_train_validation['date'] >= train_start_date)
    & (df_sales_train_validation['date'] < train_end_date)
]
val_df = df_sales_train_validation[
    (df_sales_train_validation['date'] >= val_start_date)
    & (df_sales_train_validation['date'] <= val_end_date)
]

In [ ]:
from tqdm import tqdm

In [24]:
# Define features and target variable
features = ['dept_id', 'item_id', 'cat_id', 'day_of_week', 'month', 'year', 'sales_last_week', 'sales_last_month', 'moving_avg_7', 'moving_avg_30']
target = 'value'


In [25]:
# Split the data into training and testing sets
X_train = train_df[features]
y_train = train_df[target]
X_test = val_df[features]
y_test = val_df[target]

In [26]:
# Convert categorical features to categorical data type
categorical_features = ['dept_id', 'item_id', 'cat_id', 'day_of_week', 'month', 'year']
for feature in categorical_features:
    X_train[feature] = X_train[feature].astype('category')
    X_test[feature] = X_test[feature].astype('category')


<ipython-input-26-93604b2f75bc>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[feature] = X_train[feature].astype('category')
<ipython-input-26-93604b2f75bc>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[feature] = X_test[feature].astype('category')
<ipython-input-26-93604b2f75bc>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydat

In [28]:
# Create Pool objects for training and validation
train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)

# Initialize CatBoostRegressor
model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.3,
    depth=8,
    loss_function='RMSE',
    eval_metric='MAE',
    verbose=100
)

# Train the model
model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=50)


0:	learn: 1.9970998	test: 1.8054996	best: 1.8054996 (0)	total: 3.41s	remaining: 5m 37s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.125641356
bestIteration = 40

Shrink model to first 41 iterations.


In [29]:
# Make predictions
y_pred = model.predict(X_test)

In [30]:
val_df['predicted'] = y_pred

<ipython-input-30-7071e63915fe>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df['predicted'] = y_pred


In [31]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21343 entries, 5811394 to 5832736
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   dept_id           21343 non-null  object        
 1   item_id           21343 non-null  object        
 2   cat_id            21343 non-null  object        
 3   value             21343 non-null  int64         
 4   date              21343 non-null  datetime64[ns]
 5   sales_last_week   21343 non-null  float64       
 6   sales_last_month  21343 non-null  float64       
 7   moving_avg_7      21343 non-null  float64       
 8   moving_avg_30     21343 non-null  float64       
 9   day_of_week       21343 non-null  int32         
 10  month             21343 non-null  int32         
 11  year              21343 non-null  int32         
 12  predicted         21343 non-null  float64       
dtypes: datetime64[ns](1), float64(5), int32(3), int64(1), object(3)
memory us

In [35]:
val_df["store_id"] = 'CA_3'
train_df["store_id"] = 'CA_3'

<ipython-input-35-6142d99ad8e5>:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-35-6142d99ad8e5>:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [40]:
fig = make_subplots(rows=1, cols=1)

#my_index = "FOODS_1_002_CA_3_validation"
item_id = 'FOODS_1_003'
store_id = 'CA_3'
state = 'CA'
train_df_for_plot = train_df[
            (train_df["date"] >= pd.to_datetime('2016-03-26') - timedelta(days = 100)) &
            (train_df["store_id"] == store_id) &
            (train_df["item_id"] == item_id)

        ]

val_df_for_plot = val_df[(val_df["store_id"] == store_id) &
            (val_df["item_id"] == item_id)]

fig.add_trace(
    go.Scatter(
        x=train_df_for_plot["date"],
        y=train_df_for_plot["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=val_df_for_plot["date"],
        y=val_df_for_plot["value"],
        mode="lines+markers",
        name=f"{store_id} (State: {state})"
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=val_df_for_plot["date"],
        y=val_df_for_plot["predicted"],
        mode="lines+markers",
        name="Predicted"
    ),
    row=1,
    col=1
)

fig.update_layout(
    title=f"Time Series baseline for {item_id}",
    height=400,
    showlegend=True
)

fig.show()

In [43]:
val_df['id'] = val_df["item_id"] + "_CA_3"

<ipython-input-43-8569593a93b4>:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [44]:
val_df.head()

,dept_id,item_id,cat_id,value,date,sales_last_week,sales_last_month,moving_avg_7,moving_avg_30,day_of_week,month,year,predicted,store_id,id
5811394,HOBBIES_1,HOBBIES_1_001,HOBBIES,0,2016-04-18,0.0,2.0,0.857143,1.133333,0,4,2016,0.877550,CA_3,HOBBIES_1_001_CA_3
5811395,HOBBIES_1,HOBBIES_1_002,HOBBIES,2,2016-04-18,0.0,0.0,0.285714,0.066667,0,4,2016,0.291933,CA_3,HOBBIES_1_002_CA_3
5811396,HOBBIES_1,HOBBIES_1_003,HOBBIES,0,2016-04-18,1.0,0.0,0.142857,0.266667,0,4,2016,0.143027,CA_3,HOBBIES_1_003_CA_3
5811397,HOBBIES_1,HOBBIES_1_004,HOBBIES,15,2016-04-18,5.0,0.0,7.571429,6.166667,0,4,2016,7.979574,CA_3,HOBBIES_1_004_CA_3
5811398,HOBBIES_1,HOBBIES_1_005,HOBBIES,1,2016-04-18,2.0,6.0,1.571429,2.200000,0,4,2016,1.893924,CA_3,HOBBIES_1_005_CA_3


In [47]:
metrics_by_id = val_df.groupby('id').apply(lambda group: pd.Series(evaluate(group['value'], group['predicted'])))

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24>:9: RuntimeWarning:

divide by zero encountered in scalar divide

<ipython-input-46-6d13fa4ebe24

In [48]:
val_df_for_roll_unique = val_df.drop_duplicates(subset=['id'])

metrics_by_store = metrics_by_id.groupby(val_df_for_roll_unique.set_index('id')['store_id']).apply(aggregate_metrics)
metrics_by_cat = metrics_by_id.groupby(val_df_for_roll_unique.set_index('id')['cat_id']).apply(aggregate_metrics)


In [49]:
metrics_by_store

mae       mse      rmse          mape       smape  \
store_id                                                                  
CA_3     mean    1.125642  4.582959  1.379270  1.360767e+15  123.311356   
         25%     0.437851  0.264611  0.514403  2.625856e+14   76.117092   
         median  0.783499  0.923778  0.961134  1.083147e+15  124.040495   
         75%     1.344003  2.733214  1.653243  1.873380e+15  174.997118   

                 max_error      wape  
store_id                              
CA_3     mean     2.556301       inf  
         25%      0.852819  0.611636  
         median   1.705024  0.908764  
         75%      3.095294  1.391998

In [50]:
metrics_by_cat

mae       mse      rmse          mape       smape  \
cat_id                                                                     
FOODS     mean    1.349472  5.959284  1.648698  1.424757e+15  110.153543   
          25%     0.594469  0.497765  0.705525  1.068469e+14   64.075124   
          median  0.970631  1.421037  1.192073  1.231782e+15  106.847982   
          75%     1.644121  3.862854  1.965415  2.023078e+15  154.306227   
HOBBIES   mean    0.857333  3.957465  1.062754  1.423580e+15  148.299245   
          25%     0.281662  0.133720  0.365677  4.186623e+14  113.778837   
          median  0.518687  0.450659  0.671312  9.394065e+14  161.239461   
          75%     0.892421  1.235003  1.111307  1.613055e+15  191.599482   
HOUSEHOLD mean    0.963228  3.031504  1.180287  1.239045e+15  127.885969   
          25%     0.379991  0.201854  0.449281  2.785663e+14   80.541780   
          median  0.690830  0.739170  0.859750  1.014020e+15  136.803107   
          75%     1.192032  2.028932  1.424406  1.715588e+15  180.218709   

                  max_error      wape  
cat_id                                 
FOODS     mean     3.034426       inf  
          25%      1.223335  0.552468  
          median   2.121546  0.810225  
          75%      3.648830  1.173993  
HOBBIES   mean     2.010385       inf  
          25%      0.722848  0.804030  
          median   1.302066  1.110431  
          75%      2.172913  1.838505  
HOUSEHOLD mean     2.194675       inf  
          25%      0.823672  0.625535  
          median   1.477665  0.956018  
          75%      2.631687  1.459069